## Basic Usage

### 1. Explore Available Observations

First, we'll search for Sentinel-2 observations over our area of interest within a specific time range and cloud coverage threshold.

In [6]:
# Import main classes
from wfs_sentinel_manager import AcquisitionsExplorer, DataDownloader
from sentinelhub import DataCollection, CRS


# Import all available eval scripts for Sentinel-2
from wfs_sentinel_manager.eval_scripts.s2 import (
    CCC, CWC, EVI, FAPAR, IRECI, LAI, MNDWI, MOISTURE, MSAVI2, 
    NDDI, NDMI, NDRE, NDREX, NDVI, NDWI, TrueColor, NMDI, NDCI, NDTI
)

# Define the area of interest (bounding box) - Brno, Czech Republic
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
bbox_life_es = [609366.6590792723000050,5412389.0993158128112555,616922.1317759219091386,5418025.8026246475055814]

# Define the time range for data search
time_range = '2025-08-13', '2025-08-15'

# Project name for organizing downloaded data
project_name = 'Breclav-NDVI-calculation'

# Initialize acquisition explorer with bounding box and data collection
explorer = AcquisitionsExplorer(
    bbox=bbox_life_es, 
    data_collection=DataCollection.SENTINEL2_L2A, 
    crs=CRS.UTM_33N
)

# Search for available observations within the time range
# Note: Cloud coverage is per whole scene, not just for the given bbox
explorer_result = explorer.explore(
    time_range=time_range, 
    cloud_coverage=30  # Maximum 60% cloud coverage (percentage per metadata of tile)
)

print(f"Number of acquisitions found: {len(explorer_result)}")
print(f"Acquisition dates: {explorer_result.acquisitions}")
for i, date in enumerate(explorer_result.acquisitions, 1):
    print(f"  {i}. {date}")

Number of acquisitions found: 1
Acquisition dates: (datetime.datetime(2025, 8, 14, 9, 57, 23, tzinfo=tzlocal()),)
  1. 2025-08-14 09:57:23+00:00


### (Optional) Define custom sentinel hub eval script

In [14]:
from sentinelhub import DataCollection
from wfs_sentinel_manager import AcquisitionsExplorer, DataDownloader
from wfs_sentinel_manager.eval_scripts.s2 import EvalScriptS2Index, EvalScriptS2Raw

eval_script_test = """
//VERSION=3

function setup() {
    return {
        input: ["B02"],
        output: {
            bands: 1,
            sampleType: "FLOAT32"
        }
    };
}

function evaluatePixel(sample) {
    return [sample.B02];
}
"""

class EvalScriptS2Blue(EvalScriptS2Raw):
    def __init__(self):
        super().__init__(
            eval_script_test,
            input_bands=["B02"]
        )

### 2. Download and Process Data

Now we'll download the satellite imagery and apply an evaluation script to calculate NDVI (Normalized Difference Vegetation Index).

In [15]:
# Initialize data downloader
downloader = DataDownloader(
    project_name=project_name,  # Creates a folder with this name
    resolution=10               # 10-meter resolution for Sentinel-2
)


print("Starting download and processing...")
result = downloader.download(
    explorer_result, 
    eval_script=EvalScriptS2BGR,           # Calculate NDWI
    required_valid_pixels=30    # Minimum valid pixels needed (percentage per polygon)
)

print(f"Download completed successfully!")
result

Starting download and processing...
Download completed successfully!


[{'timestamp': datetime.datetime(2025, 8, 14, 9, 57, 23, tzinfo=tzlocal()),
  'valid_pixels': 100,
  'tiff_path': 'data/Breclav-NDVI-calculation/EvalScriptS2BGR/2025-08-14_09-57-23.tiff'}]

### 3. (Optional) Move output to {repo_root}/{output_folder} ---

In [16]:
from pathlib import Path
import os, subprocess, shutil
from typing import Optional

# === Config: where results should end up (relative to project root) ===
output_folder = Path("adaptation-strategy-breclav/data/LandCover/Sentinel_2/BGR")

# ---------- helpers ----------
def resolve_project_root() -> Path:
    """
    Robustly resolve your project root.
    Order:
      1) PROJECT_ROOT or LIFE_2025_ROOT env var
      2) Git top-level (if inside a repo)
      3) Nearest ancestor that has both 'scripts/' and 'data/' dirs
      4) Current working directory (last resort)
    """
    env_root = os.environ.get("PROJECT_ROOT") or os.environ.get("LIFE_2025_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            return p

    try:
        r = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, check=False
        )
        if r.returncode == 0 and r.stdout.strip():
            g = Path(r.stdout.strip()).resolve()
            if g.exists():
                return g
    except Exception:
        pass

    here = Path.cwd().resolve()
    for anc in [here] + list(here.parents):
        if (anc / "scripts").is_dir() and (anc / "data").is_dir():
            return anc

    return here


def find_nearest_data_dir(start: Optional[Path] = None) -> Path:
    """
    Find the nearest 'data' directory starting from CWD upward.
    If none found, prefer './data' next to CWD (create if necessary).
    """
    base = (start or Path.cwd()).resolve()
    for anc in [base] + list(base.parents):
        cand = anc / "data"
        if cand.is_dir():
            return cand
    # fallback: assume local ./data
    cand = base / "data"
    cand.mkdir(parents=True, exist_ok=True)
    return cand


def find_leaf_output_dir(data_root: Path) -> Path:
    """
    Return the deepest non-empty directory under data_root.
    Raises if nothing non-empty is found.
    """
    leaf: Optional[Path] = None
    max_depth = -1

    for p in data_root.rglob("*"):
        if p.is_dir():
            try:
                has_files = any(child.is_file() for child in p.iterdir())
            except PermissionError:
                has_files = False
            if has_files:
                depth = len(p.relative_to(data_root).parts)
                if depth > max_depth:
                    max_depth = depth
                    leaf = p

    if leaf is None:
        raise FileNotFoundError(f"No non-empty output folder found under {data_root}")
    return leaf


def move_merge(src_dir: Path, dst_dir: Path) -> None:
    """
    Merge-move contents of src_dir into dst_dir.
    Directories are merged; files are moved (overwrite if same name).
    """
    dst_dir.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        target = dst_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
            shutil.rmtree(item)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                target.unlink()
            shutil.move(str(item), str(target))


def prune_empty_dirs(path: Path, stop_at: Path) -> None:
    """
    Remove empty directories from 'path' up to (and including) stop_at if empty.
    """
    cur = path
    while True:
        try:
            if cur.exists() and not any(cur.iterdir()):
                cur.rmdir()
        except Exception:
            break
        if cur == stop_at:
            break
        if stop_at not in cur.parents:
            break
        cur = cur.parent


# ---------- run move ----------
PROJECT_ROOT = resolve_project_root()
LOCAL_DATA = find_nearest_data_dir()
SRC_LEAF = find_leaf_output_dir(LOCAL_DATA)
DEST = (PROJECT_ROOT / output_folder).resolve()

move_merge(SRC_LEAF, DEST)
prune_empty_dirs(SRC_LEAF, LOCAL_DATA)

print(f"Moved outputs to: {DEST}")

Moved outputs to: /home/sagemaker-user/notebooks/adaptation-strategy-breclav/data/LandCover/Sentinel_2/BGR


In [ ]:
from pathlib import Path
import rasterio
from rasterio.merge import merge
import os, subprocess

# --------- USER CONFIG ----------
YEAR = 2023
LAI_ROOT_REL = Path("life-2025/data/ECOSYSTEM_SERVICES/air_filtration/LAI")
FILL_VALUE = -999.0
# --------------------------------

def resolve_project_root() -> Path:
    env_root = os.environ.get("PROJECT_ROOT") or os.environ.get("LIFE_2025_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            return p
    try:
        r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=False)
        if r.returncode == 0 and r.stdout.strip():
            g = Path(r.stdout.strip()).resolve()
            if g.exists():
                return g
    except Exception:
        pass
    here = Path.cwd().resolve()
    for anc in [here] + list(here.parents):
        if (anc / "scripts").is_dir() and (anc / "data").is_dir():
            return anc
    return here

# ---------- RUN: merge exported annual means ----------
PROJECT_ROOT = resolve_project_root()
LAI_ROOT = (PROJECT_ROOT / LAI_ROOT_REL).resolve()

# ✅ correct folder (matches your log)
IN_DIR = LAI_ROOT / f"{YEAR}_annual_mean"

tifs = sorted(list(IN_DIR.glob("*.tif")) + list(IN_DIR.glob("*.tiff")))
if not tifs:
    raise FileNotFoundError(f"No exported annual-mean rasters found in: {IN_DIR}")

srcs = [rasterio.open(p) for p in tifs]

try:
    mosaic, out_transform = merge(srcs, nodata=float(FILL_VALUE), method="first")

    out_profile = srcs[0].profile.copy()
    out_profile.update(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=out_transform,
        count=1,
        dtype="float32",
        nodata=float(FILL_VALUE),
        compress="deflate",
        predictor=2
    )

    # ✅ output directly under LAI
    out_path = LAI_ROOT / f"LAI_all_localities_annual_avg_{YEAR}.tif"

    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(mosaic[0].astype("float32"), 1)

    print(f"[OK] Merged mosaic written: {out_path}")

    with rasterio.open(out_path, "r") as ds:
        print("OUT transform:", ds.transform)
        print("OUT bounds:", ds.bounds)
        print("OUT nodata:", ds.nodata)

finally:
    for s in srcs:
        s.close()


### (Optional) Define custom sentinel hub eval script

In [5]:
from sentinelhub import DataCollection
from wfs_sentinel_manager import AcquisitionsExplorer, DataDownloader
from wfs_sentinel_manager.eval_scripts.s2 import EvalScriptS2Index, EvalScriptS2Raw

eval_script = """
//VERSION=3
//This script was converted from v1 to v3 using the converter API

var degToRad = Math.PI / 180;

function evaluatePixel(samples) {
  if (isValidVeg(samples.SCL)) {
      var sample = samples;
      return [0];
  } else return [-999];
}

function setup() {
  return {
    input: [{
      bands: [
        "SCL",
        "B03",
        "B04",
      ]
    }],
    output: [
        {          
          sampleType: SampleType.FLOAT32,
          bands: 1
        }
    ]
  }
}

function isValidVeg (sampl) {
    if ((sampl == 4)||(sampl == 5)||(sampl == 6)) {
         return true;
  } else return false;
}

function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
    outputMetadata.userData = { "norm_factor":  inputMetadata.normalizationFactor }
}
"""

# Define a custom eval script class to be passed to DataDownloader via the eval_script parameter
class MyCustomS2Script(EvalScriptS2Raw):
    def __init__(self):
        super(MyCustomS2Script, self).__init__(eval_script,no_data_value=-999, clouds_value=-999)

In [ ]:
# Import main classes
from wfs_sentinel_manager import AcquisitionsExplorer, DataDownloader
from sentinelhub import DataCollection, CRS


# Import all available eval scripts for Sentinel-2
from wfs_sentinel_manager.eval_scripts.s2 import (
    CCC, CWC, EVI, FAPAR, IRECI, LAI, MNDWI, MOISTURE, MSAVI2, 
    NDDI, NDMI, NDRE, NDREX, NDVI, NDWI, TrueColor
)

# Define the area of interest (bounding box) - Brno, Czech Republic
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
bbox_life_es = [631897.3035999999847263,5392581.8618999999016523,642875.5260999996680766,5408145.2434999998658895]

# Define the time range for data search
time_range = '2025-03-01', '2025-11-30'

# Project name for organizing downloaded data
project_name = 'Breclav-NDVI-calculation'

# Initialize acquisition explorer with bounding box and data collection
explorer = AcquisitionsExplorer(
    bbox=bbox_life_es, 
    data_collection=DataCollection.SENTINEL2_L2A, 
    crs=CRS.UTM_33N
)

# Search for available observations within the time range
# Note: Cloud coverage is per whole scene, not just for the given bbox
explorer_result = explorer.explore(
    time_range=time_range, 
    cloud_coverage=50  # Maximum 60% cloud coverage (percentage per metadata of tile)
)

print(f"Number of acquisitions found: {len(explorer_result)}")
print(f"Acquisition dates: {explorer_result.acquisitions}")
for i, date in enumerate(explorer_result.acquisitions, 1):
    print(f"  {i}. {date}")

In [11]:
from sentinelhub import DataCollection
from wfs_sentinel_manager import AcquisitionsExplorer, DataDownloader
from wfs_sentinel_manager.eval_scripts.s2 import EvalScriptS2Index, EvalScriptS2Raw

eval_script = """
//VERSION=3
//This script was converted from v1 to v3 using the converter API

var degToRad = Math.PI / 180;

function evaluatePixel(samples) {
  if (isValidVeg(samples.SCL)) {
      var sample = samples;
      return [0];
  } else return [-999];
}

function setup() {
  return {
    input: [{
      bands: [
        "SCL",
        "B03",
        "B04",
      ]
    }],
    output: [
        {          
          sampleType: SampleType.FLOAT32,
          bands: 1
        }
    ]
  }
}

function isValidVeg (sampl) {
    if ((sampl == 4)||(sampl == 5)||(sampl == 6)) {
         return true;
  } else return false;
}

function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
    outputMetadata.userData = { "norm_factor":  inputMetadata.normalizationFactor }
}
"""

# Define a custom eval script class to be passed to DataDownloader via the eval_script parameter
class MyCustomS2Script(EvalScriptS2Raw):
    def __init__(self):
        super(MyCustomS2Script, self).__init__(eval_script,no_data_value=-999, clouds_value=-999)

In [6]:
# Initialize data downloader
downloader = DataDownloader(
    project_name=project_name,  # Creates a folder with this name
    resolution=10               # 10-meter resolution for Sentinel-2
)


print("Starting download and processing...")
result = downloader.download(
    explorer_result, 
    eval_script=LAI,           # Calculate NDWI
    required_valid_pixels=30    # Minimum valid pixels needed (percentage per polygon)
)

print(f"Download completed successfully!")
result

Starting download and processing...


/home/sagemaker-user/.conda/envs/sentinel-manager/lib/python3.13/site-packages/sentinelhub/download/sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
/home/sagemaker-user/.conda/envs/sentinel-manager/lib/python3.13/site-packages/sentinelhub/download/sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
/home/sagemaker-user/.conda/envs/sentinel-manager/lib/python3.13/site-packages/sentinelhub/download/sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
/home/sagemaker-user/.conda/envs/sentinel-manager/lib/python3.13/site-packages/sentinelhub/download/sentinelhub_client.py:93: SHRateLimitWarning: Download rate limit hit
  warnings.warn("Download rate limit hit", category=SHRateLimitWarning)
/home/sagemaker-user/.conda/envs

Download completed successfully!


[{'timestamp': datetime.datetime(2025, 9, 28, 9, 57, 4, tzinfo=tzlocal()),
  'valid_pixels': 62,
  'tiff_path': 'data/Breclav-LAI-calculation/LAI/2025-09-28_09-57-04.tiff'},
 {'timestamp': datetime.datetime(2025, 9, 21, 10, 6, 59, tzinfo=tzlocal()),
  'valid_pixels': 100,
  'tiff_path': 'data/Breclav-LAI-calculation/LAI/2025-09-21_10-06-59.tiff'},
 {'timestamp': datetime.datetime(2025, 9, 15, 9, 57, 17, tzinfo=tzlocal()),
  'valid_pixels': 98,
  'tiff_path': 'data/Breclav-LAI-calculation/LAI/2025-09-15_09-57-17.tiff'},
 {'timestamp': datetime.datetime(2025, 9, 11, 10, 6, 59, tzinfo=tzlocal()),
  'valid_pixels': 83,
  'tiff_path': 'data/Breclav-LAI-calculation/LAI/2025-09-11_10-06-59.tiff'},
 {'timestamp': datetime.datetime(2025, 9, 3, 9, 57, 22, tzinfo=tzlocal()),
  'valid_pixels': 78,
  'tiff_path': 'data/Breclav-LAI-calculation/LAI/2025-09-03_09-57-22.tiff'},
 {'timestamp': datetime.datetime(2025, 9, 1, 10, 7, 1, tzinfo=tzlocal()),
  'valid_pixels': 100,
  'tiff_path': 'data/Breclav-

In [7]:
from pathlib import Path
import os, subprocess, shutil
from typing import Optional

# === Config: where results should end up (relative to project root) ===
output_folder = Path("adaptation-strategy-breclav/data/Sentinel_2/LAI/2025/")

# ---------- helpers ----------
def resolve_project_root() -> Path:
    """
    Robustly resolve your project root.
    Order:
      1) PROJECT_ROOT or LIFE_2025_ROOT env var
      2) Git top-level (if inside a repo)
      3) Nearest ancestor that has both 'scripts/' and 'data/' dirs
      4) Current working directory (last resort)
    """
    env_root = os.environ.get("PROJECT_ROOT") or os.environ.get("LIFE_2025_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if p.exists():
            return p

    try:
        r = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, check=False
        )
        if r.returncode == 0 and r.stdout.strip():
            g = Path(r.stdout.strip()).resolve()
            if g.exists():
                return g
    except Exception:
        pass

    here = Path.cwd().resolve()
    for anc in [here] + list(here.parents):
        if (anc / "scripts").is_dir() and (anc / "data").is_dir():
            return anc

    return here


def find_nearest_data_dir(start: Optional[Path] = None) -> Path:
    """
    Find the nearest 'data' directory starting from CWD upward.
    If none found, prefer './data' next to CWD (create if necessary).
    """
    base = (start or Path.cwd()).resolve()
    for anc in [base] + list(base.parents):
        cand = anc / "data"
        if cand.is_dir():
            return cand
    # fallback: assume local ./data
    cand = base / "data"
    cand.mkdir(parents=True, exist_ok=True)
    return cand


def find_leaf_output_dir(data_root: Path) -> Path:
    """
    Return the deepest non-empty directory under data_root.
    Raises if nothing non-empty is found.
    """
    leaf: Optional[Path] = None
    max_depth = -1

    for p in data_root.rglob("*"):
        if p.is_dir():
            try:
                has_files = any(child.is_file() for child in p.iterdir())
            except PermissionError:
                has_files = False
            if has_files:
                depth = len(p.relative_to(data_root).parts)
                if depth > max_depth:
                    max_depth = depth
                    leaf = p

    if leaf is None:
        raise FileNotFoundError(f"No non-empty output folder found under {data_root}")
    return leaf


def move_merge(src_dir: Path, dst_dir: Path) -> None:
    """
    Merge-move contents of src_dir into dst_dir.
    Directories are merged; files are moved (overwrite if same name).
    """
    dst_dir.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        target = dst_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
            shutil.rmtree(item)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                target.unlink()
            shutil.move(str(item), str(target))


def prune_empty_dirs(path: Path, stop_at: Path) -> None:
    """
    Remove empty directories from 'path' up to (and including) stop_at if empty.
    """
    cur = path
    while True:
        try:
            if cur.exists() and not any(cur.iterdir()):
                cur.rmdir()
        except Exception:
            break
        if cur == stop_at:
            break
        if stop_at not in cur.parents:
            break
        cur = cur.parent


# ---------- run move ----------
PROJECT_ROOT = resolve_project_root()
LOCAL_DATA = find_nearest_data_dir()
SRC_LEAF = find_leaf_output_dir(LOCAL_DATA)
DEST = (PROJECT_ROOT / output_folder).resolve()

move_merge(SRC_LEAF, DEST)
prune_empty_dirs(SRC_LEAF, LOCAL_DATA)

print(f"Moved outputs to: {DEST}")

Moved outputs to: /home/sagemaker-user/notebooks/adaptation-strategy-breclav/data/Sentinel_2/LAI/2025
